In [ ]:
# Import necessary libraries
import os

import matplotlib.pyplot as plt
import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()


# Database connection setup function
def connect_to_db():
    global db_connection, cursor
    try:
        db_connection = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME"),
        )
        cursor = db_connection.cursor(dictionary=True)
        print("Database connection established successfully.")
    except mysql.connector.Error as e:
        print(f"Error connecting to database: {e}")
        db_connection = None
        cursor = None


# Establish the database connection
connect_to_db()

In [ ]:
# Cell 2: Query the data and calculate metrics
# Check if the connection is successful
if db_connection:
    # Create a cursor object to execute SQL queries
    cursor = db_connection.cursor()

    # Query to get the count of all entries
    count_query = "SELECT COUNT(*) AS total_entries FROM DNSResults"
    cursor.execute(count_query)
    total_entries = cursor.fetchone()[0]  # Use integer index to get the count

    # Query to get the oldest and newest timestamps
    timestamp_query = """
    SELECT MIN(db_timestamp) AS oldest_entry, MAX(db_timestamp) AS newest_entry
    FROM DNSResults
    """
    cursor.execute(timestamp_query)
    result = cursor.fetchone()
    oldest_entry = result[0]  # Use integer index for the oldest entry
    newest_entry = result[1]  # Use integer index for the newest entry

    # Ensure 'db_timestamp' is in datetime format
    oldest_entry = pd.to_datetime(oldest_entry)
    newest_entry = pd.to_datetime(newest_entry)

    print(f"Oldest Entry: {oldest_entry}")
    print(f"Newest Entry: {newest_entry}")

    # Calculate the time range in minutes
    time_range = (newest_entry - oldest_entry).total_seconds() / 60
    print(f"Time Range: {time_range} minutes")
    time_range_hours = time_range / 60
    print(f"Time Range: {time_range_hours:,.2f} hours")

    # Calculate requests solved per minute
    requests_per_minute = total_entries / time_range
    print(f"Requests solved per minute: {requests_per_minute:,.2f}")

    # Calculate requests done per 24 hours
    requests_per_24_hours = requests_per_minute * 60 * 24
    print(f"Requests done per 24 hours: {requests_per_24_hours:,.2f}")

    # Total requests solved yet:
    print(f"Total requests solved yet: {total_entries:,}")

else:
    print("Failed to establish database connection.")

In [ ]:
# Cell 3: Optional - Plotting the number of requests over time
if db_connection:
    # Set 'db_timestamp' as index for time series plotting
    df.set_index("db_timestamp", inplace=True)

    # Resample by minute and count the number of requests
    df_resampled = df.resample("1T").size()

    # Plot the number of requests over time
    plt.figure(figsize=(12, 6))
    plt.plot(
        df_resampled.index,
        df_resampled.values,
        label="Requests per Minute",
        color="blue",
    )
    plt.xlabel("Time")
    plt.ylabel("Number of Requests")
    plt.title("Requests Over Time")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Failed to establish database connection.")

In [ ]:
if cursor:
    cursor.close()
if db_connection:
    db_connection.close()
    print("Database connection closed.")